In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# การระบุตัวเลือก Sampler

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="เวอร์ชันแพ็กเกจ">

โค้ดในหน้านี้พัฒนาโดยใช้ข้อกำหนดต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
คุณสามารถใช้ตัวเลือกเพื่อปรับแต่ง Sampler primitive ได้ ส่วนนี้จะเน้นวิธีระบุตัวเลือก Qiskit Runtime primitive อินเทอร์เฟซของเมธอด `run()` ของ primitive นั้นเหมือนกันทุกการใช้งาน แต่ตัวเลือกไม่ใช่ ดูข้อมูลเกี่ยวกับตัวเลือก [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) และ [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html) ได้จาก API reference ที่เกี่ยวข้อง

<span id="pass-options"></span>
## ตั้งค่าตัวเลือก Sampler
คุณตั้งค่าตัวเลือกได้เมื่อเริ่มต้น Sampler หลังจากเริ่มต้น Sampler หรืออัปเดตตัวเลือกหลังจากที่ Sampler ถูกเริ่มต้นแล้ว ดูคำแนะนำการใช้เทคนิคเหล่านี้ได้ที่หัวข้อ [บทนำสู่ตัวเลือก](/guides/runtime-options-overview#options-precedence)

นอกจากนี้ คุณสามารถตั้งค่า `shots` ในเมธอด `run()` ได้ตามที่อธิบายในส่วนถัดไป
<span id="run-method"></span>
### เมธอด Run()
ค่าเดียวที่สามารถส่งไปยัง `run()` คือค่าที่กำหนดไว้ในอินเทอร์เฟซ นั่นคือ `shots` ซึ่งจะเขียนทับค่าที่ตั้งไว้สำหรับ `default_shots` ในการรันปัจจุบัน

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

Example:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### กรณีพิเศษ
<span id="shots"></span>
#### Shots
เมธอด `SamplerV2.run` รับอาร์กิวเมนต์สองตัว ได้แก่ รายการ PUB ซึ่งแต่ละตัวสามารถระบุค่า shots เฉพาะ PUB และอาร์กิวเมนต์คีย์เวิร์ด shots ค่า shots เหล่านี้เป็นส่วนหนึ่งของอินเทอร์เฟซการดำเนินการ Sampler และเป็นอิสระจากตัวเลือกของ Runtime Sampler ค่าเหล่านี้มีลำดับความสำคัญสูงกว่าค่าที่ระบุเป็นตัวเลือก เพื่อให้สอดคล้องกับการ abstract ของ Sampler

อย่างไรก็ตาม หาก `shots` ไม่ได้ถูกระบุโดย PUB ใดหรือในอาร์กิวเมนต์คีย์เวิร์ดของ run (หรือถ้าทั้งหมดเป็น `None`) ค่า shots จากตัวเลือกจะถูกใช้ โดยเฉพาะ `default_shots`

สรุปแล้ว นี่คือลำดับความสำคัญในการระบุ shots ใน Sampler สำหรับ PUB ใด ๆ:

1. ถ้า PUB ระบุ shots ให้ใช้ค่านั้น
2. ถ้าอาร์กิวเมนต์คีย์เวิร์ด `shots` ถูกระบุใน `run` ให้ใช้ค่านั้น
4. ถ้า `twirling` เปิดใช้งาน (True ตามค่าเริ่มต้น) จะใช้ผลคูณของ `num_randomizations` และ `shots_per_randomization` ตามที่ระบุเป็น[ตัวเลือก `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
5. ถ้า `sampler.options.default_shots` ถูกระบุ ให้ใช้ค่านั้น

ดังนั้น ถ้า shots ถูกระบุในทุกตำแหน่งที่เป็นไปได้ ค่าที่มีลำดับความสำคัญสูงสุด (shots ที่ระบุใน PUB) จะถูกใช้

ตัวอย่าง: